In [1]:
from pathlib import Path
from datetime import datetime
import json
import shutil
import sys
import subprocess
import pandas as pd

direktori_aktif = Path.cwd()
direktori_project = direktori_aktif.parent if direktori_aktif.name.lower() == "notebooks" else direktori_aktif

direktori_src = direktori_project / "src"
direktori_outputs = direktori_project / "reports" / "outputs"
direktori_examples = direktori_project / "examples"
direktori_intelligence = direktori_project / "data" / "intelligence"

lokasi_engine_v5 = direktori_src / "phishrisk_engine_v5.py"
lokasi_cli_v5 = direktori_src / "run_phishrisk_v5.py"
lokasi_trusted = direktori_intelligence / "trusted_safe_domains_global.csv"

file_wajib = [lokasi_engine_v5, lokasi_cli_v5, lokasi_trusted]

validasi_awal = pd.DataFrame([{
    "nama_file": file.name,
    "lokasi": str(file),
    "tersedia": file.exists(),
    "ukuran_kb": round(file.stat().st_size / 1024, 2) if file.exists() else 0,
} for file in file_wajib])

display(validasi_awal)

if not validasi_awal["tersedia"].all():
    raise FileNotFoundError("Ada file wajib yang belum tersedia.")

print("Semua file wajib tersedia.")


,nama_file,lokasi,tersedia,ukuran_kb
0,phishrisk_engine_v5.py,C:\Users\ASUS\PHISHING\src\phishrisk_engine_v5.py,True,26.65
1,run_phishrisk_v5.py,C:\Users\ASUS\PHISHING\src\run_phishrisk_v5.py,True,3.27
2,trusted_safe_domains_global.csv,C:\Users\ASUS\PHISHING\data\intelligence\trust...,True,1.00


Semua file wajib tersedia.


In [2]:
isi = lokasi_engine_v5.read_text(encoding="utf-8")

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
backup = direktori_src / f"phishrisk_engine_v5_backup_before_huggingface_fix_{timestamp}.py"
shutil.copy2(lokasi_engine_v5, backup)

perubahan = []

guard_lama = "if is_trusted_safe and suspicious_score == 0 and not lookalike_detected and not uses_punycode:"
guard_baru = "if is_trusted_safe and suspicious_score == 0 and not uses_punycode:"

if guard_lama in isi:
    isi = isi.replace(guard_lama, guard_baru)
    perubahan.append("Guard trusted safe tidak lagi bergantung pada lookalike_detected.")

lama_suppress = '''                skor_final = min(skor_final, 24)
                brand_but_not_official = 0
                alasan.append("Domain masuk daftar trusted safe dan tidak memiliki sinyal phishing kuat.")
'''

baru_suppress = '''                skor_final = min(skor_final, 24)
                brand_but_not_official = 0
                lookalike_detected = 0
                alasan.append("Domain masuk daftar trusted safe dan tidak memiliki sinyal phishing kuat.")
'''

if lama_suppress in isi and "lookalike_detected = 0" not in isi:
    isi = isi.replace(lama_suppress, baru_suppress)
    perubahan.append("Lookalike ikut disuppress untuk trusted safe domain netral.")

lama_intel = '''            intelligence_kuat = (
                "tiruan_brand_berisiko" in intelligence_status
                or "domain_mirip_brand" in intelligence_status
                or "kata_mencurigakan_tinggi" in intelligence_status
            )
'''

baru_intel = '''            intelligence_kuat = (
                "tiruan_brand_berisiko" in intelligence_status
                or "domain_mirip_brand_berisiko" in intelligence_status
                or "kata_mencurigakan_tinggi" in intelligence_status
            )
'''

if lama_intel in isi:
    isi = isi.replace(lama_intel, baru_intel)
    perubahan.append("brand_tidak_resmi_perlu_tinjauan tidak dianggap hard intelligence untuk trusted safe domain.")

lokasi_engine_v5.write_text(isi, encoding="utf-8")

print("Patch HuggingFace false positive selesai.")
print("Backup:", backup)
print("Jumlah perubahan:", len(perubahan))
for item in perubahan:
    print("-", item)

if len(perubahan) == 0:
    print("Tidak ada perubahan baru. Kemungkinan patch sudah pernah diterapkan.")


Patch HuggingFace false positive selesai.
Backup: C:\Users\ASUS\PHISHING\src\phishrisk_engine_v5_backup_before_huggingface_fix_20260522_233917.py
Jumlah perubahan: 3
- Guard trusted safe tidak lagi bergantung pada lookalike_detected.
- Lookalike ikut disuppress untuk trusted safe domain netral.
- brand_tidak_resmi_perlu_tinjauan tidak dianggap hard intelligence untuk trusted safe domain.


In [3]:
if str(direktori_src) not in sys.path:
    sys.path.insert(0, str(direktori_src))

import importlib
import phishrisk_engine_v5

importlib.reload(phishrisk_engine_v5)

engine = phishrisk_engine_v5.PhishRiskEngineV5(
    direktori_project=direktori_project,
    prefer_model="best",
)

url_uji = [
    "https://huggingface.co",
    "https://pandas.pydata.org",
    "https://python.org",
    "https://scikit-learn.org",
    "https://kaggle.com",
    "https://wikipedia.org",
    "https://huggingface.co/login-update",
    "https://pandas.pydata.org/account-verify",
    "http://bca-login-update.test",
    "http://micros0ft-login-update.test",
    "https://xn--micrsoft-q4a.test",
]

hasil = []

for url in url_uji:
    item = engine.analisis_url(url)
    hasil.append(item)

data_hasil = pd.DataFrame(hasil)

def expected_status(url):
    if url in [
        "https://huggingface.co",
        "https://pandas.pydata.org",
        "https://python.org",
        "https://scikit-learn.org",
        "https://kaggle.com",
        "https://wikipedia.org",
    ]:
        return "Terlihat Aman"

    if "login-update" in url or "account-verify" in url:
        return "Perlu Tinjauan/Berisiko"

    return "Berisiko"

data_hasil["expected"] = data_hasil["url"].apply(expected_status)

def status_uji(row):
    expected = row["expected"]
    hasil = row["hasil_akhir_v5"]

    if expected == "Terlihat Aman":
        return "lolos" if hasil == "Terlihat Aman" else "gagal"

    if expected == "Berisiko":
        return "lolos" if hasil == "Berisiko" else "gagal"

    return "lolos" if hasil in ["Perlu Tinjauan", "Berisiko"] else "gagal"

data_hasil["status_uji"] = data_hasil.apply(status_uji, axis=1)

lokasi_hasil_fix = direktori_outputs / "hasil_fix_huggingface_false_positive_engine_v5_step17b.csv"
data_hasil.to_csv(lokasi_hasil_fix, index=False, encoding="utf-8")

kolom_tampil = [
    "url", "expected", "trusted_safe_domain", "skor_model_v5", "skor_final_v5",
    "kategori_risiko_v5", "hasil_akhir_v5", "intelligence_status", "alasan_v5", "status_uji"
]
kolom_tampil = [kolom for kolom in kolom_tampil if kolom in data_hasil.columns]

print("Uji fix selesai:", lokasi_hasil_fix)
display(data_hasil[kolom_tampil])


Uji fix selesai: C:\Users\ASUS\PHISHING\reports\outputs\hasil_fix_huggingface_false_positive_engine_v5_step17b.csv


,url,expected,trusted_safe_domain,skor_model_v5,skor_final_v5,kategori_risiko_v5,hasil_akhir_v5,intelligence_status,alasan_v5,status_uji
0,https://huggingface.co,Terlihat Aman,1,4.18,4.18,Rendah,Terlihat Aman,brand_tidak_resmi_perlu_tinjauan,Domain masuk daftar trusted safe dan tidak mem...,lolos
1,https://pandas.pydata.org,Terlihat Aman,1,72.55,24.00,Rendah,Terlihat Aman,belum_ada_sinyal_kuat,Domain masuk daftar trusted safe dan tidak mem...,lolos
2,https://python.org,Terlihat Aman,1,0.63,0.63,Rendah,Terlihat Aman,belum_ada_sinyal_kuat,Domain masuk daftar trusted safe dan tidak mem...,lolos
3,https://scikit-learn.org,Terlihat Aman,1,2.37,2.37,Rendah,Terlihat Aman,belum_ada_sinyal_kuat,Domain masuk daftar trusted safe dan tidak mem...,lolos
4,https://kaggle.com,Terlihat Aman,1,1.90,1.90,Rendah,Terlihat Aman,belum_ada_sinyal_kuat,Domain masuk daftar trusted safe dan tidak mem...,lolos
5,https://wikipedia.org,Terlihat Aman,1,0.58,0.58,Rendah,Terlihat Aman,belum_ada_sinyal_kuat,Domain masuk daftar trusted safe dan tidak mem...,lolos
6,https://huggingface.co/login-update,Perlu Tinjauan/Berisiko,1,36.27,92.00,Sangat Tinggi,Berisiko,tiruan_brand_berisiko,URL memakai nama brand tetapi bukan domain res...,lolos
7,https://pandas.pydata.org/account-verify,Perlu Tinjauan/Berisiko,1,7.30,72.00,Tinggi,Berisiko,kata_mencurigakan_tinggi,URL mengandung kata yang sering muncul pada se...,lolos
8,http://bca-login-update.test,Perlu Tinjauan/Berisiko,0,91.91,92.00,Sangat Tinggi,Berisiko,tiruan_brand_berisiko,URL memakai nama brand tetapi bukan domain res...,lolos
9,http://micros0ft-login-update.test,Perlu Tinjauan/Berisiko,0,90.26,92.00,Sangat Tinggi,Berisiko,tiruan_brand_berisiko,URL memakai nama brand tetapi bukan domain res...,lolos


In [4]:
input_cli = direktori_examples / "input_url_step17b_fix_huggingface.csv"
output_cli = direktori_outputs / "hasil_cli_step17b_fix_huggingface.csv"

pd.DataFrame({"url": url_uji}).to_csv(input_cli, index=False, encoding="utf-8")

perintah = [
    sys.executable,
    str(lokasi_cli_v5),
    "--mode", "urls",
    "--input", str(input_cli),
    "--url-column", "url",
    "--output", str(output_cli),
    "--model-mode", "best",
]

hasil_cli = subprocess.run(perintah, capture_output=True, text=True)

print("Return code:", hasil_cli.returncode)
print("STDOUT:")
print(hasil_cli.stdout)
print("STDERR:")
print(hasil_cli.stderr)

if hasil_cli.returncode != 0:
    raise RuntimeError("CLI gagal setelah HuggingFace fix.")

data_cli = pd.read_csv(output_cli)

kolom_cli = [
    "url", "trusted_safe_domain", "skor_final_v5",
    "kategori_risiko_v5", "hasil_akhir_v5", "engine_version"
]
kolom_cli = [kolom for kolom in kolom_cli if kolom in data_cli.columns]

display(data_cli[kolom_cli])


Return code: 0
STDOUT:
PhishRisk Engine V5 selesai.
Mode: urls
Output: C:\Users\ASUS\PHISHING\reports\outputs\hasil_cli_step17b_fix_huggingface.csv
hasil_akhir_v5 kategori_risiko_v5  skor_final_v5 engine_version
 Terlihat Aman             Rendah           4.18             V5
 Terlihat Aman             Rendah          24.00             V5
 Terlihat Aman             Rendah           0.63             V5
 Terlihat Aman             Rendah           2.37             V5
 Terlihat Aman             Rendah           1.90             V5
 Terlihat Aman             Rendah           0.58             V5
      Berisiko      Sangat Tinggi          92.00             V5
      Berisiko             Tinggi          72.00             V5
      Berisiko      Sangat Tinggi          92.00             V5
      Berisiko      Sangat Tinggi          92.00             V5

STDERR:



,url,trusted_safe_domain,skor_final_v5,kategori_risiko_v5,hasil_akhir_v5,engine_version
0,https://huggingface.co,1,4.18,Rendah,Terlihat Aman,V5
1,https://pandas.pydata.org,1,24.00,Rendah,Terlihat Aman,V5
2,https://python.org,1,0.63,Rendah,Terlihat Aman,V5
3,https://scikit-learn.org,1,2.37,Rendah,Terlihat Aman,V5
4,https://kaggle.com,1,1.90,Rendah,Terlihat Aman,V5
5,https://wikipedia.org,1,0.58,Rendah,Terlihat Aman,V5
6,https://huggingface.co/login-update,1,92.00,Sangat Tinggi,Berisiko,V5
7,https://pandas.pydata.org/account-verify,1,72.00,Tinggi,Berisiko,V5
8,http://bca-login-update.test,0,92.00,Sangat Tinggi,Berisiko,V5
9,http://micros0ft-login-update.test,0,92.00,Sangat Tinggi,Berisiko,V5


In [5]:
jumlah_gagal = int((data_hasil["status_uji"] == "gagal").sum())
status_fix = "fix_siap" if jumlah_gagal == 0 and hasil_cli.returncode == 0 else "perlu_tinjauan"

data_status = pd.DataFrame([{
    "status_fix": status_fix,
    "jumlah_uji_gagal": jumlah_gagal,
    "cli_return_code": int(hasil_cli.returncode),
    "catatan": (
        "HuggingFace false positive sudah diperbaiki. Engine V5 siap lanjut integrasi Streamlit."
        if status_fix == "fix_siap"
        else "Masih ada hasil yang perlu ditinjau."
    ),
}])

lokasi_status_fix = direktori_outputs / "status_fix_huggingface_false_positive_engine_v5_step17b.csv"
data_status.to_csv(lokasi_status_fix, index=False, encoding="utf-8")

metadata = {
    "nama_notebook": "17B_FIX_huggingface_false_positive_engine_v5.ipynb",
    "nama_tahap": "Fix HuggingFace False Positive Engine V5",
    "status": status_fix,
    "engine_patch_file": str(lokasi_engine_v5),
    "engine_backup_file": str(backup),
    "hasil_fix": str(lokasi_hasil_fix),
    "hasil_cli": str(output_cli),
    "status_fix": str(lokasi_status_fix),
    "catatan": [
        "Fix ini menyempurnakan patch trusted safe domain.",
        "Trusted safe domain netral dapat turun risiko meski terdeteksi brand ringan.",
        "Sinyal kuat seperti login/update/verify, punycode, dan domain tiruan tetap tidak dibuat aman.",
    ],
}

lokasi_metadata = direktori_outputs / "metadata_step17b_fix_huggingface_false_positive_engine_v5.json"
lokasi_metadata.write_text(json.dumps(metadata, indent=4, ensure_ascii=False), encoding="utf-8")

print("Status fix:", lokasi_status_fix)
print("Metadata:", lokasi_metadata)
display(data_status)


Status fix: C:\Users\ASUS\PHISHING\reports\outputs\status_fix_huggingface_false_positive_engine_v5_step17b.csv
Metadata: C:\Users\ASUS\PHISHING\reports\outputs\metadata_step17b_fix_huggingface_false_positive_engine_v5.json


,status_fix,jumlah_uji_gagal,cli_return_code,catatan
0,fix_siap,0,0,HuggingFace false positive sudah diperbaiki. E...
